<h2>File paths and imports</h2>

This steps is important because it tells the program where it can access the files needed throughout the process.

You must specify 3 paths:

<li> <b>model_path</b>: body detection model
<li> <b>input_video_directory</b>: Folder containing the input videos
<li> <b>output_video_directory</b>: Folder where the treated videos will be redirected to

You can also mention videos that you don't want to be treated. To do so, simply indicate their name(s) without the extension in <b>ignore_S1</b> and <b>ignore_S2</b>.

In [1]:
from ui_lib import *

# path of the body detection model
model_path = "/home/thomas-rixen/Documents/UCL/Master2/Master-thesis/local/ChimpRec/Code/Tracking/Manual Correction/Body_detection_model.pt"

# video paths:
# input video directory (without any annotation)
input_video_directory = "/home/thomas-rixen/Documents/UCL/Master2/Master-thesis/local/ChimpRec_videos/input/"
# output (final version - with human interaction)
output_video_directory = "/home/thomas-rixen/Documents/UCL/Master2/Master-thesis/local/ChimpRec_videos/ouput/"

ignore_S1 = [ # related to step 1
    "example_1",
    "example_2"
]

ignore_S2 = [ # related to step 2
    "example_1",
    "example_2"
]

/home/thomas-rixen/Documents/UCL/Master2/Master-thesis/local/ChimpRec/.venv/lib/python3.10/site-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


<h2>First step:</h2>

This step will process the input videos automatically. In other words, it will draw rectangles around each individuals and track them throughout the video.

In [2]:
#FIRST STEP CODE (double-click to extend)

# directory containing all the manual modifications
mannual_annotations_directory = f"{input_video_directory}/manual_annotations"
output_video_directory_temp = f"{output_video_directory}/temp"
raw_text_output_directory = f"{output_video_directory_temp}/raw_output"

# Create the directories if they do not exist yet
os.makedirs(input_video_directory, exist_ok=True)
os.makedirs(output_video_directory, exist_ok=True)
os.makedirs(mannual_annotations_directory, exist_ok=True)
os.makedirs(output_video_directory_temp, exist_ok=True)
os.makedirs(raw_text_output_directory, exist_ok=True)

max_cosine_distance = 0.5       # maximal distance to match an object (lower = more strict)
nn_budget = None                # maximal buffer size
metric = nn_matching.NearestNeighborDistanceMetric("cosine", max_cosine_distance, nn_budget)

# YOLOv8s initialisation
YOLOv8s = YOLO(model_path)

# DeepSORT initialisation
DeepSort = DeepSortTracker(metric)

# Osnet initialisation
Osnet = torchreid.models.build_model(name='osnet_x1_0', num_classes=751, pretrained=True)
Osnet.eval()

for input_video in os.listdir(input_video_directory):
    if input_video.endswith(".mp4") or input_video.endswith(".MP4"):

        full_video_path = os.path.join(input_video_directory, input_video)
        video_name = os.path.splitext(input_video)[0]

        if video_name in ignore_S1:
            print(f"{video_name}.mp4 ignored")
            continue

        # creation of the manual_annotation textfile if it doesn't exist yet
        annotation_file_path = f"{mannual_annotations_directory}/{video_name}.txt"
        try:
            with open(annotation_file_path, 'x') as f:
                print(f"{video_name}.txt automatically created in {mannual_annotations_directory}.")
        except FileExistsError:
            print(f"{video_name}.txt already present in {mannual_annotations_directory}.")
        print("\n")

        # production of the textual outputs
        perform_tracking(
            input_video_path = full_video_path, 
            output_text_file_path = f"{raw_text_output_directory}/{video_name}.txt", 
            detection_model = YOLOv8s, 
            tracker = DeepSort,
            confidence_threshold = 0.5, 
            model_feature_extraction = Osnet
        )
        print(f"Annotations ready for video: {full_video_path}.\n")

        # production of the visual output
        draw_bbox_from_file(
            file_path = f"{raw_text_output_directory}/{video_name}.txt", 
            input_video_path = full_video_path, 
            output_video_path = f"{output_video_directory_temp}/{video_name}-(temp).mp4",
            annotation_type="bbox",
            draw_frame_count=True
        )
        print(f"Treatment done: {full_video_path}.\n")

Downloading...
From: https://drive.google.com/uc?id=1LaG1EJpHrxdAxKnSCJ_i0u-nbxSAeiFY
To: /home/thomas-rixen/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth
100%|██████████| 10.9M/10.9M [00:02<00:00, 4.26MB/s]


Successfully loaded imagenet pretrained weights from "/home/thomas-rixen/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
20241015 - 12h41.txt automatically created in /home/thomas-rixen/Documents/UCL/Master2/Master-thesis/local/ChimpRec_videos/input//manual_annotations.




Tracking progress (20241015 - 12h41.MP4):   2%|▏         | 258/10536 [02:25<1:36:36,  1.77it/s]


KeyboardInterrupt: 

<h2>Second step:</h2>

This final step will take into account your modifications to modify the output of the automated process.

<b>If you need to modify annotations previously created:</b> simply run this part of the code. In this case, there's no need to run the above cells.

In [ ]:
#SECOND STEP CODE (double-click to extend)

for input_video in os.listdir(input_video_directory):
    if input_video.endswith(".mp4") or input_video.endswith(".MP4"):
        full_video_path = os.path.join(input_video_directory, input_video)
        video_name = os.path.splitext(input_video)[0]

        if video_name in ignore_S2:
            print(f"{video_name}.mp4 ignored")
            continue

        annotation_file = f"{mannual_annotations_directory}/{video_name}.txt"

        raw_reader = raw_tracking_data_reader(f"{raw_text_output_directory}/{video_name}.txt")

        try:
            edit_reader = modification_reader(annotation_file)
        except:
            print(f"Error: the manual annotation file related to the video <{full_video_path}> is not found. It must be located at <{annotation_file}>.")
            continue
        
        metadata_file_path = f"{output_video_directory}/{video_name}-treated.txt"
        output_video_path = f"{output_video_directory}/{video_name}-treated.mp4"
        writer = data_writer(metadata_file_path)

        # computation of the new metadata file
        modified_data = edit_raw_output(raw_reader, edit_reader) 

        # production of the textual output
        writer.write(modified_data)

        # production of the visual output
        draw_bbox_from_file(
            file_path = metadata_file_path, 
            input_video_path = full_video_path, 
            output_video_path = output_video_path,
            annotation_type="triangle"
        )
        print(f"Treatment done: {full_video_path}.\n")